In [ ]:
from neo4j import GraphDatabase
import pandas as pd

URI = "neo4j://127.0.0.1:7687"
AUTH_USER = "neo4j"
AUTH_PASSWORD = "master2025"
DATABASE = "llmagraphtrkg"

driver = GraphDatabase.driver(URI, auth=(AUTH_USER, AUTH_PASSWORD))

In [ ]:
from langchain_neo4j import Neo4jGraph
graph = Neo4jGraph(
    url="neo4j://127.0.0.1:7687",
    username="neo4j",
    password="master2025",
    database="llmagraphtrkg",
)

In [ ]:
import pandas as pd

community_stats = graph.query("""
MATCH (e:Entity)
WHERE e.communities IS NOT NULL
WITH last(e.communities) AS communityId
RETURN communityId, count(*) AS nodeCount
ORDER BY nodeCount DESC
""")

df_comm = pd.DataFrame(community_stats)
df_comm.head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Theme ---
sns.set_theme(style="white")

# --- Schriftarten & Farben (klar schwarz) ---
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.titlecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

# --- Farben (professionelle neutrale Palette) ---
neutral_palette = ["#bdbdbd", "#636363"]  # Hellgrau / Dunkelgrau


In [ ]:
plt.figure(figsize=(4, 3))

sns.barplot(
    data=df_comm,
    x="communityId",
    y="nodeCount",
    palette=neutral_palette
)

plt.title("Community Size Distribution")
plt.xlabel("Community ID")
plt.ylabel("Node Count")
plt.xticks(rotation=0)

# Achsen schwarz machen
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_color("black")

plt.tight_layout()
plt.show()


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

target_community = 5

edges = graph.query("""
MATCH (e1:Entity)-[:SIMILAR]-(e2:Entity)
WHERE last(e1.communities) = $cid
  AND last(e2.communities) = $cid
RETURN id(e1) AS source, id(e2) AS target
LIMIT 200
""", params={"cid": target_community})

nodes = graph.query("""
MATCH (e:Entity)
WHERE last(e.communities) = $cid
RETURN id(e) AS node_id, e.id AS label
LIMIT 200
""", params={"cid": target_community})

df_edges = pd.DataFrame(edges)
df_nodes = pd.DataFrame(nodes)

G = nx.Graph()
for _, row in df_nodes.iterrows():
    G.add_node(row["node_id"], label=row["label"])

for _, row in df_edges.iterrows():
    G.add_edge(row["source"], row["target"])

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G)
nx.draw(G, pos, with_labels=False, node_size=50)
# Labels optional:
# labels = {n: d["label"] for n, d in G.nodes(data=True)}
# nx.draw_networkx_labels(G, pos, labels, font_size=6)
plt.title(f"Community {target_community} – SIMILAR graph")
plt.axis("off")
plt.show()


In [ ]:
from neo4j import GraphDatabase
import pandas as pd

URI = "neo4j://127.0.0.1:7687"
AUTH_USER = "neo4j"
AUTH_PASSWORD = "master2025"
DATABASE = "llmakg"

driver = GraphDatabase.driver(URI, auth=(AUTH_USER, AUTH_PASSWORD))

In [ ]:

COUNT_QUERY = """
RETURN
  count{ (:Document) }   AS documents,
  count{ (:Chunk) }      AS chunks,
  count{ (:__Entity__) } AS entities
"""

REL_COUNT_QUERY = """
MATCH ()-[r]->()
RETURN count(r) AS relationships
"""

In [ ]:



with driver.session(database=DATABASE) as session:
    row = session.run(COUNT_QUERY).single()
    rel_row = session.run(REL_COUNT_QUERY).single()

stats_before = pd.DataFrame([{
    "stage": "before",
    "documents": row["documents"],
    "chunks": row["chunks"],
    "entities": row["entities"],
    "relationships": rel_row["relationships"],
}])

stats_before.to_csv("kg_stats_before_llmaindex.csv", index=False)
stats_before


In [ ]:
with driver.session(database=DATABASE) as session:
    row = session.run(COUNT_QUERY).single()
    rel_row = session.run(REL_COUNT_QUERY).single()

stats_after = pd.DataFrame([{
    "stage": "after",
    "documents": row["documents"],
    "chunks": row["chunks"],
    "entities": row["entities"],
    "relationships": rel_row["relationships"],
}])
stats_after

In [ ]:
stats_before = pd.read_csv("kg_stats_before_llmaindex.csv")
stats_all = pd.concat([stats_before, stats_after], ignore_index=True)
stats_all


In [ ]:
with driver.session(database=DATABASE) as session:
    row = session.run(COUNT_QUERY).single()
    rel_row = session.run(REL_COUNT_QUERY).single()

stats_after = pd.DataFrame([{
    "stage": "after",
    "documents": row["documents"],
    "chunks": row["chunks"],
    "entities": row["entities"],
    "relationships": rel_row["relationships"],
}])
stats_after

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Theme ---
sns.set_theme(style="white")

# --- Schriftarten & Farben (klar schwarz) ---
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.titlecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

# --- Farben (professionelle neutrale Palette) ---
neutral_palette = ["#bdbdbd", "#636363"]  # Hellgrau / Dunkelgrau

plt.figure(figsize=(4, 3))

sns.barplot(
    data=stats_long,
    x="type",          # entities / relationships
    y="count",
    hue="stage",       # BEFORE vs AFTER KG-Build
    palette=neutral_palette
)

plt.title("Entity & Relationship Counts Before vs After Resolution")
plt.xlabel("Element Type")
plt.ylabel("Count")
plt.xticks(rotation=0)

# Achsen schwarz machen
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_color("black")

# Legende anpassen
legend = ax.legend(title="Stage")
plt.setp(legend.get_texts(), color="black")
plt.setp(legend.get_title(), color="black")

plt.tight_layout()
plt.show()
